# L2L-Arbor Integration

This notebook implements a full L2L-Arbor integration following the L2L conventions and project structure.

## Overview
1. **Project Structure Setup** - Following L2L conventions
2. **Custom Optimizee Implementation** - NeuronOptimizee class
3. **Experiment Management** - L2L Experiment configuration
4. **Optimizer Configuration** - Evolution Strategies and Cross-Entropy
5. **Trajectory Management** - Parameter organization and result analysis

In [1]:
# L2L
#!pip install git+https://github.com/Meta-optimization/L2L.git

## Stage 1: Project Directory Structure Setup

Following L2L conventions for project organization

In [2]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

# Arbor imports
import arbor as A
from arbor import units as U

# Check if L2L is available
try:
    from l2l.optimizees.optimizee import Optimizee
    from l2l.utils.experiment import Experiment
    from l2l.utils.trajectory import Trajectory
    L2L_AVAILABLE = True
    print("✓ L2L is installed and available")
except ImportError:
    L2L_AVAILABLE = False
    print("⚠ L2L is not installed. Using mock classes for demonstration")
    # Mock classes for demonstration
    class Optimizee:
        def __init__(self, traj):
            self.traj = traj
    class Experiment:
        pass
    class Trajectory:
        pass

✓ L2L is installed and available


In [3]:
# Project directory structure (following L2L conventions)
PROJECT_ROOT = Path.cwd()
L2L_PROJECT_DIR = PROJECT_ROOT / "l2l_arbor_project"

# Standard L2L directory structure
DIRS = {
    'results': L2L_PROJECT_DIR / "results",
    'logs': L2L_PROJECT_DIR / "logs",
    'configs': L2L_PROJECT_DIR / "configs",
    'optimizees': L2L_PROJECT_DIR / "optimizees",
    'scripts': L2L_PROJECT_DIR / "scripts",
    'data': L2L_PROJECT_DIR / "data",
    'analysis': L2L_PROJECT_DIR / "analysis"
}

def setup_project_structure():
    """Create L2L project directory structure"""
    for name, directory in DIRS.items():
        directory.mkdir(parents=True, exist_ok=True)
        print(f"✓ Created {name}: {directory}")
    
    # Create __init__.py files for Python modules
    for module_dir in ['optimizees', 'scripts', 'analysis']:
        init_file = DIRS[module_dir] / "__init__.py"
        init_file.touch(exist_ok=True)
    
    print("\nProject structure created successfully!")
    return DIRS

# Setup the structure
project_dirs = setup_project_structure()

✓ Created results: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results
✓ Created logs: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/logs
✓ Created configs: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/configs
✓ Created optimizees: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/optimizees
✓ Created scripts: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/scripts
✓ Created data: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/data
✓ Created analysis: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/analysis

Project structure created successfully!


## Stage 2: Custom Optimizee Implementation for Neuron Models

### 2.1 Create NeuronOptimizee Class

In [4]:
class NeuronOptimizee(Optimizee):
    """
    Custom Optimizee for neuron model optimization.
    Inherits from L2L Optimizee base class and follows L2L interface specifications.
    """
    
    def __init__(self, traj, morphology_file='single_cell_allen.swc',
                 reference_file='single_cell_allen_neuron_ref.csv',
                 fit_file='single_cell_allen_fit.json'):
        """
        Initialize the NeuronOptimizee.
        
        Args:
            traj: L2L Trajectory object
            morphology_file: Path to SWC morphology file
            reference_file: Path to reference voltage data
            fit_file: Path to Allen parameter file
        """
        super().__init__(traj)
        
        # Load neuron data
        self._load_neuron_data(morphology_file, reference_file, fit_file)
        
        # 2.2 Define neuron parameter space
        self._define_parameter_space()
        
        # 5.1 Parameter organization - implement hierarchical parameter groups
        self._organize_trajectory_parameters(traj)
        
        # Initialize fitness evaluator
        self.fitness_evaluator = None
        
        print("NeuronOptimizee initialized successfully")
    
    def _load_neuron_data(self, morphology_file, reference_file, fit_file):
        """Load neuron morphology and reference data"""
        # Load morphology
        try:
            raw = A.load_swc_neuron(morphology_file)
            self.morphology = raw.morphology
            self.labels = raw.labels
            self.labels.add_swc_tags()
        except Exception as e:
            print(f"Warning: Could not load morphology: {e}")
            self.morphology = None
            self.labels = None
        
        # Load reference data
        try:
            self.ref_data = pd.read_csv(reference_file, usecols=[1,2], index_col=0)
            with open(fit_file) as f:
                self.fit_data = json.load(f)
            
            # Process reference data
            self.ref_data['U/mV'] *= 1000.0
            self.ref_data['U/mV'] += self.fit_data["fitting"][0]["junction_potential"]
            
            # Extract spike times
            self._extract_reference_features()
        except Exception as e:
            print(f"Warning: Could not load reference data: {e}")
            self.ref_data = None
            self.fit_data = None
    
    def _extract_reference_features(self):
        """Extract electrophysiological features from reference data"""
        if self.ref_data is None:
            self.ref_features = {}
            return
        
        # Extract spike times
        threshold = -10  # mV
        voltage = self.ref_data['U/mV'].values
        time = self.ref_data.index.values
        
        # Find threshold crossings
        crossings = np.where((voltage[:-1] < threshold) & (voltage[1:] > threshold))[0]
        spike_times = time[crossings + 1]
        
        # Calculate additional features
        self.ref_features = {
            'spike_times': spike_times,
            'spike_count': len(spike_times),
            'mean_frequency': len(spike_times) / (time[-1] - time[0]) * 1000 if len(spike_times) > 0 else 0,
            'voltage_trace': voltage,
            'time_trace': time
        }
        
        print(f"Extracted reference features: {len(spike_times)} spikes")
    
    def _define_parameter_space(self):
        """
        2.2 Define neuron parameter space
        Define parameter groups for different neuron attributes
        """
        self.parameter_groups = {
            # Membrane parameters
            'membrane': {
                'cm': {'min': 0.5, 'max': 2.0, 'default': 1.0, 'unit': 'uF/cm2', 'description': 'Membrane capacitance'},
                'Ra': {'min': 50, 'max': 300, 'default': 100, 'unit': 'Ohm.cm', 'description': 'Axial resistance'},
                'Vm': {'min': -90, 'max': -60, 'default': -70, 'unit': 'mV', 'description': 'Initial membrane potential'},
            },
            
            # Channel parameters (based on Allen data)
            'channels': {
                'gbar_NaV': {'min': 0.03, 'max': 0.07, 'default': 0.05, 'unit': 'S/cm2', 'description': 'Sodium conductance'},
                'gbar_Kv3_1': {'min': 0.15, 'max': 0.7, 'default': 0.3, 'unit': 'S/cm2', 'description': 'Potassium conductance'},
                'gbar_Ca_HVA': {'min': 5e-5, 'max': 2e-4, 'default': 1e-4, 'unit': 'S/cm2', 'description': 'HVA calcium conductance'},
                'gbar_Ca_LVA': {'min': 0.001, 'max': 0.01, 'default': 0.005, 'unit': 'S/cm2', 'description': 'LVA calcium conductance'},
            },
            
            # Passive parameters
            'passive': {
                'g_pas': {'min': 0.00005, 'max': 0.002, 'default': 0.0001, 'unit': 'S/cm2', 'description': 'Passive conductance'},
                'e_pas': {'min': -90, 'max': -60, 'default': -70, 'unit': 'mV', 'description': 'Passive reversal potential'},
            },
            
            # Dynamics parameters
            'dynamics': {
                'gamma_CaDynamics': {'min': 0.01, 'max': 0.05, 'default': 0.02, 'unit': '', 'description': 'Calcium removal rate'},
                'decay_CaDynamics': {'min': 20, 'max': 300, 'default': 100, 'unit': 'ms', 'description': 'Calcium decay time'},
            }
        }
        
        # Flatten parameter space for easy access
        self.flat_params = {}
        for group_name, group_params in self.parameter_groups.items():
            for param_name, config in group_params.items():
                full_name = f"{group_name}.{param_name}"
                self.flat_params[full_name] = config
        
        print(f"Defined {len(self.flat_params)} parameters across {len(self.parameter_groups)} groups")
    
    def _organize_trajectory_parameters(self, traj):
        """
        5.1 Parameter organization
        Implement hierarchical parameter groups for neuron attributes
        """
        # Add main parameter group
        traj.f_add_parameter_group('individual', 'Individual neuron parameters')
        
        # L2L has issues with nested groups, so we'll use flat parameter names
        # but keep the logical grouping in our internal structure
        for full_name, config in self.flat_params.items():
            # Replace dots with underscores for L2L compatibility
            param_name = full_name.replace('.', '_')
            traj.f_add_parameter_to_group(
                'individual',
                param_name,
                config['default']
            )
        
        # Add result groups for organization
        traj.f_add_result_group('fitness', 'Fitness values')
        traj.f_add_result_group('features', 'Electrophysiological features')
        traj.f_add_result_group('simulation', 'Simulation outputs')
        
        print("Trajectory parameters organized")
    
    def create_individual(self):
        """
        2.1 Implement required create_individual() method
        Generate neuron parameter sets
        
        Returns:
            dict: Parameter dictionary with random values within bounds
        """
        individual = {}
        
        # Generate random parameters within bounds
        for param_name, config in self.flat_params.items():
            # Use uniform distribution for initial population
            value = np.random.uniform(config['min'], config['max'])
            # Use underscore version for L2L
            individual[param_name.replace('.', '_')] = value
        
        # Parameter validation
        individual = self._validate_parameters(individual)
        
        return individual
    
    def _validate_parameters(self, params):
        """
        2.2 Implement parameter validation method
        Ensure biophysical plausibility
        """
        validated = params.copy()
        
        # Convert back to dot notation for validation
        dot_params = {}
        for key, value in validated.items():
            dot_key = key.replace('_', '.')
            # Check if this matches our expected format
            if dot_key in self.flat_params:
                dot_params[dot_key] = value
            else:
                # Try to find the correct mapping
                for flat_key in self.flat_params:
                    if flat_key.replace('.', '_') == key:
                        dot_params[flat_key] = value
                        break
        
        # Example constraints:
        # 1. Ensure sodium conductance > potassium conductance in some regions
        if 'channels.gbar_NaV' in dot_params and 'channels.gbar_Kv3_1' in dot_params:
            if dot_params['channels.gbar_NaV'] < dot_params['channels.gbar_Kv3_1'] * 0.8:
                # Swap or adjust
                dot_params['channels.gbar_NaV'] = dot_params['channels.gbar_Kv3_1'] * 1.2
        
        # 2. Ensure passive reversal potential is below spike threshold
        if 'passive.e_pas' in dot_params:
            dot_params['passive.e_pas'] = min(dot_params['passive.e_pas'], -65)
        
        # Convert back to underscore notation
        result = {}
        for key, value in dot_params.items():
            result[key.replace('.', '_')] = value
        
        return result
    
    def bounding_func(self, individual):
        """
        2.2 Set biophysical plausibility bounds and constraints
        """
        bounded = {}
        
        for param_name, value in individual.items():
            # Convert underscore to dot notation to check bounds
            dot_name = param_name.replace('_', '.')
            # Try to find matching parameter
            config = None
            if dot_name in self.flat_params:
                config = self.flat_params[dot_name]
            else:
                # Search for correct mapping
                for flat_key in self.flat_params:
                    if flat_key.replace('.', '_') == param_name:
                        config = self.flat_params[flat_key]
                        break
            
            if config:
                # Clip to bounds
                bounded[param_name] = np.clip(value, config['min'], config['max'])
            else:
                bounded[param_name] = value
        
        # Apply validation
        bounded = self._validate_parameters(bounded)
        
        return bounded
    
    def simulate(self, traj):
        """
        2.1 Implement simulate() method to evaluate neuron model fitness
        2.3 Implement fitness evaluation
        
        Args:
            traj: L2L Trajectory containing current parameters
            
        Returns:
            tuple: (fitness,) as required by L2L
        """
        # Get parameters from trajectory
        params = self._get_parameters_from_trajectory(traj)
        
        try:
            # Run neuron simulation
            sim_results = self._run_neuron_simulation(params)
            
            # 2.3 Extract features from simulation results
            extracted_features = self._extract_simulation_features(sim_results)
            
            # 2.3 Compare simulation vs target features
            fitness = self._calculate_fitness(extracted_features, self.ref_features)
            
            # 5.2 Store results and metadata
            self._store_simulation_results(traj, sim_results, extracted_features, fitness)
            
        except Exception as e:
            # 2.3 Handle simulation failures
            print(f"Simulation failed: {e}")
            fitness = self._get_penalty_fitness()
            traj.f_add_result('simulation.error', str(e))
            traj.f_add_result('fitness.value', fitness)
        
        return (fitness,)  # L2L requires tuple
    
    def _get_parameters_from_trajectory(self, traj):
        """Extract parameters from L2L trajectory"""
        params = {}
        
        # Get all parameters from the individual group
        for param_name in dir(traj.individual):
            if not param_name.startswith('_'):
                # Convert underscore back to dot notation
                dot_name = param_name.replace('_', '.')
                if dot_name in self.flat_params:
                    params[dot_name] = getattr(traj.individual, param_name)
        
        return params
    
    def _run_neuron_simulation(self, params):
        """Run Arbor simulation with given parameters"""
        # This is a placeholder - in real implementation, create and run Arbor model
        # For demonstration, return mock results
        
        if self.morphology is None:
            # Mock simulation
            time = np.linspace(0, 1000, 10000)
            voltage = -70 + 10 * np.random.randn(10000)
            spike_times = np.sort(np.random.uniform(200, 800, np.random.randint(2, 8)))
        else:
            # TODO: Implement actual Arbor simulation
            time = np.linspace(0, 1000, 10000)
            voltage = -70 + 10 * np.random.randn(10000)
            spike_times = np.sort(np.random.uniform(200, 800, 4))
        
        return {
            'time': time,
            'voltage': voltage,
            'spike_times': spike_times,
            'parameters': params
        }
    
    def _extract_simulation_features(self, sim_results):
        """
        2.3 Implement feature extraction from simulation results
        Define target electrophysiological features
        """
        spike_times = sim_results['spike_times']
        voltage = sim_results['voltage']
        time = sim_results['time']
        
        features = {
            'spike_times': spike_times,
            'spike_count': len(spike_times),
            'mean_frequency': len(spike_times) / (time[-1] - time[0]) * 1000 if len(spike_times) > 0 else 0,
        }
        
        # Additional features
        if len(spike_times) > 1:
            isis = np.diff(spike_times)
            features['mean_isi'] = np.mean(isis)
            features['cv_isi'] = np.std(isis) / np.mean(isis) if np.mean(isis) > 0 else 0
        else:
            features['mean_isi'] = 0
            features['cv_isi'] = 0
        
        # Voltage features
        features['resting_potential'] = np.mean(voltage[:1000])  # First 100ms
        features['voltage_variance'] = np.var(voltage)
        
        return features
    
    def _calculate_fitness(self, sim_features, ref_features):
        """
        2.3 Create fitness function comparing simulation vs target features
        """
        if not ref_features:
            # If no reference, use default targets
            ref_features = {
                'spike_count': 4,
                'mean_frequency': 4.0,
                'mean_isi': 250.0
            }
        
        fitness = 0.0
        
        # Spike count error (most important)
        spike_count_error = abs(sim_features['spike_count'] - ref_features['spike_count'])
        fitness += 10.0 * spike_count_error
        
        # Spike timing error
        if 'spike_times' in ref_features and len(sim_features['spike_times']) > 0:
            # Match closest spikes
            timing_errors = []
            for ref_spike in ref_features['spike_times']:
                if len(sim_features['spike_times']) > 0:
                    min_error = min(abs(sim_spike - ref_spike) 
                                  for sim_spike in sim_features['spike_times'])
                    timing_errors.append(min_error)
            if timing_errors:
                fitness += np.mean(timing_errors)
        
        # Frequency error
        freq_error = abs(sim_features['mean_frequency'] - ref_features.get('mean_frequency', 4.0))
        fitness += 5.0 * freq_error
        
        return fitness
    
    def _get_penalty_fitness(self):
        """2.3 Handle simulation failures and parameter validation errors"""
        return 10000.0  # Large penalty
    
    def _store_simulation_results(self, traj, sim_results, features, fitness):
        """
        5.2 Result storage and analysis
        Store electrophysiological features, fitness values, and simulation metadata
        """
        # Store fitness
        traj.f_add_result('fitness.value', fitness)
        
        # Store features
        for feature_name, value in features.items():
            if isinstance(value, (int, float)):
                traj.f_add_result(f'features.{feature_name}', value)
        
        # Store spike times separately
        traj.f_add_result('features.spike_times_array', features['spike_times'])
        
        # Store simulation metadata
        traj.f_add_result('simulation.timestamp', datetime.now().isoformat())
        traj.f_add_result('simulation.duration', sim_results['time'][-1])
        
        # Store parameters used (for tracking)
        traj.f_add_result('simulation.parameters', sim_results['parameters'])

## Stage 3: Experiment Management Setup

### 3.1 Configure Experiment Class

In [10]:
def setup_experiment(experiment_name="neuron_optimization", 
                    results_dir=None,
                    log_stdout=True,
                    continuable=True,
                    add_timestamp=True):
    """
    3.1 Configure Experiment class
    Initialize with appropriate root directory and parameters
    """
    if not L2L_AVAILABLE:
        print("L2L not available. Returning mock objects.")
        return None, None
    
    # Use default results directory if not specified
    if results_dir is None:
        results_dir = DIRS['results']
    
    # Add timestamp to experiment name to avoid conflicts
    if add_timestamp:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        experiment_name = f"{experiment_name}_{timestamp}"
    
    # Create experiment
    experiment = Experiment(
        root_dir_path=str(results_dir)
    )
    
    # Prepare experiment with appropriate parameters
    try:
        traj, environment = experiment.prepare_experiment(
            name=experiment_name,
            log_stdout=log_stdout,  # Set up logging and stdout redirection
            continuable=continuable,  # Allow resuming
            automatic_storing=True,
            comment="L2L-Arbor neuron model optimization experiment"
        )
    except Exception as e:
        if "already exsiting outputfiles" in str(e):
            print(f"Warning: Experiment directory already exists. Using timestamp.")
            # Force timestamp
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            experiment_name = f"{experiment_name}_{timestamp}"
            traj, environment = experiment.prepare_experiment(
                name=experiment_name,
                log_stdout=log_stdout,
                continuable=continuable,
                automatic_storing=True,
                comment="L2L-Arbor neuron model optimization experiment"
            )
        else:
            raise
    
    # Store experiment metadata as results instead of config
    # L2L trajectories use config differently than expected
    traj.f_add_result('experiment_info.name', experiment_name)
    traj.f_add_result('experiment_info.timestamp', datetime.now().isoformat())
    traj.f_add_result('experiment_info.results_dir', str(results_dir))
    
    print(f"Experiment '{experiment_name}' configured successfully")
    print(f"Results will be stored in: {results_dir}/{experiment_name}")
    
    return experiment, traj

# Test experiment setup
if L2L_AVAILABLE:
    test_exp, test_traj = setup_experiment("test_neuron_opt")
    if test_exp:
        print("✓ Experiment setup successful")

All output logs can be found in directory  /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_neuron_opt_20250711_001620/logs
MainProcess utils.trajectory zq-Lenovo-Legion-R7000P2021H 239442 INFO    : Added new parameter group: runner_params
Runner parameters used: {'srun': '', 'exec': 'python3 "/home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_neuron_opt_20250711_001620/simulation/run_optimizee.py"', 'max_workers': 32, 'work_path': '/home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_neuron_opt_20250711_001620', 'paths_obj': <l2l.paths.Paths object at 0x7996af36eba0>}
Experiment 'test_neuron_opt_20250711_001620' configured successfully
Results will be stored in: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_neuron_opt_20250711_001620
✓ Experiment setup successful


## Stage 4: Optimizer Selection and Configuration

### 4.1 Evolution Strategies Implementation

In [11]:
def configure_evolution_strategy(traj, optimizee, 
                               population_size=50,
                               learning_rate=0.1,
                               noise_std=0.1):
    """
    4.1 Evolution strategies implementation
    Configure evolution strategies optimizer for neuron parameter optimization
    """
    if not L2L_AVAILABLE:
        print("L2L not available")
        return None
    
    try:
        from l2l.optimizers.evolutionstrategies import EvolutionStrategiesOptimizer, EvolutionStrategiesParameters
    except ImportError:
        print("Evolution Strategies optimizer not available")
        return None
    
    # Set appropriate population size and mutation parameters for neuron models
    parameters = EvolutionStrategiesParameters(
        learning_rate=learning_rate,
        noise_std=noise_std,
        mirrored_sampling=True,  # Use mirrored sampling for better gradient estimation
        fitness_shaping=True,  # Use fitness shaping to handle outliers
        n_roll_outs=population_size,  # ES uses n_roll_outs for population size
    )
    
    # Create optimizer
    optimizer = EvolutionStrategiesOptimizer(
        traj,
        optimizee_create_individual=optimizee.create_individual,
        optimizee_fitness_weights=(-1,),  # Minimize fitness
        parameters=parameters,
        optimizee_bounding_func=optimizee.bounding_func
    )
    
    # Store optimizer info as results
    traj.f_add_result('optimizer_info.type', 'evolution_strategies')
    traj.f_add_result('optimizer_info.population_size', population_size)
    traj.f_add_result('optimizer_info.learning_rate', learning_rate)
    traj.f_add_result('optimizer_info.noise_std', noise_std)
    
    print(f"Evolution Strategies optimizer configured with population size {population_size}")
    return optimizer

### 4.2 Cross-Entropy Method Implementation

In [12]:
def configure_cross_entropy(traj, optimizee,
                          population_size=100,
                          elite_fraction=0.2,
                          smoothing=0.7):
    """
    4.2 Cross-entropy method implementation
    Configure cross-entropy optimizer as alternative optimization method
    """
    if not L2L_AVAILABLE:
        print("L2L not available")
        return None
    
    try:
        from l2l.optimizers.crossentropy import CrossEntropyOptimizer, CrossEntropyParameters
        from l2l.optimizers.crossentropy.distribution import NormalDistribution
    except ImportError:
        print("Cross-Entropy optimizer not available")
        return None
    
    # Get number of parameters
    dummy_individual = optimizee.create_individual()
    n_params = len(dummy_individual)
    
    # Set elite selection criteria and distribution update parameters
    parameters = CrossEntropyParameters(
        pop_size=population_size,
        elite_frac=elite_fraction,  # Top 20% are elite
        smoothing=smoothing,  # Smoothing parameter for distribution update
        temp_decay=0.0,  # No temperature decay
        n_iteration=100,  # Max iterations
        distribution=NormalDistribution(n_params, noise_std=0.1),
        stop_criterion=np.inf
    )
    
    # Create optimizer
    optimizer = CrossEntropyOptimizer(
        traj,
        optimizee_create_individual=optimizee.create_individual,
        optimizee_fitness_weights=(-1,),  # Minimize
        parameters=parameters,
        optimizee_bounding_func=optimizee.bounding_func
    )
    
    # Store optimizer info as results
    traj.f_add_result('optimizer_info.type', 'cross_entropy')
    traj.f_add_result('optimizer_info.population_size', population_size)
    traj.f_add_result('optimizer_info.elite_fraction', elite_fraction)
    traj.f_add_result('optimizer_info.smoothing', smoothing)
    
    print(f"Cross-Entropy optimizer configured with population size {population_size}")
    return optimizer

### 4.3 Optimizer Parameter Tuning

In [13]:
def create_optimizer_configs():
    """
    4.3 Optimizer parameter tuning
    Create parameter configuration files for different optimizer settings
    """
    configs = {
        'evolution_strategies': {
            'exploration': {
                'population_size': 100,
                'learning_rate': 0.2,
                'noise_std': 0.2,
                'description': 'High exploration for initial search'
            },
            'exploitation': {
                'population_size': 50,
                'learning_rate': 0.05,
                'noise_std': 0.05,
                'description': 'Fine-tuning with low noise'
            },
            'balanced': {
                'population_size': 75,
                'learning_rate': 0.1,
                'noise_std': 0.1,
                'description': 'Balanced exploration-exploitation'
            }
        },
        'cross_entropy': {
            'conservative': {
                'population_size': 150,
                'elite_fraction': 0.1,
                'smoothing': 0.9,
                'description': 'Conservative updates with small elite'
            },
            'aggressive': {
                'population_size': 50,
                'elite_fraction': 0.3,
                'smoothing': 0.5,
                'description': 'Aggressive updates with large elite'
            },
            'standard': {
                'population_size': 100,
                'elite_fraction': 0.2,
                'smoothing': 0.7,
                'description': 'Standard configuration'
            }
        }
    }
    
    # Save configurations
    config_file = DIRS['configs'] / 'optimizer_configs.json'
    with open(config_file, 'w') as f:
        json.dump(configs, f, indent=2)
    
    print(f"Optimizer configurations saved to {config_file}")
    return configs

# Create and save configurations
optimizer_configs = create_optimizer_configs()

Optimizer configurations saved to /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/configs/optimizer_configs.json


In [14]:
def select_optimizer(traj, optimizee, optimizer_type='evolution_strategies', 
                    config_name='balanced'):
    """
    4.3 Implement optimizer selection based on problem characteristics
    """
    # Load configurations
    config_file = DIRS['configs'] / 'optimizer_configs.json'
    if config_file.exists():
        with open(config_file) as f:
            configs = json.load(f)
    else:
        configs = create_optimizer_configs()
    
    # Get configuration
    if optimizer_type not in configs:
        print(f"Unknown optimizer type: {optimizer_type}")
        return None
    
    if config_name not in configs[optimizer_type]:
        print(f"Unknown config: {config_name}")
        config_name = 'balanced' if optimizer_type == 'evolution_strategies' else 'standard'
    
    config = configs[optimizer_type][config_name]
    print(f"\nUsing {optimizer_type} with '{config_name}' configuration:")
    print(f"  {config['description']}")
    
    # Create optimizer
    if optimizer_type == 'evolution_strategies':
        return configure_evolution_strategy(
            traj, optimizee,
            population_size=config['population_size'],
            learning_rate=config['learning_rate'],
            noise_std=config['noise_std']
        )
    elif optimizer_type == 'cross_entropy':
        return configure_cross_entropy(
            traj, optimizee,
            population_size=config['population_size'],
            elite_fraction=config['elite_fraction'],
            smoothing=config['smoothing']
        )
    else:
        print(f"Optimizer {optimizer_type} not implemented")
        return None

## Stage 5: Trajectory Management for Neuron Data

### 5.2 Result Storage and Analysis

In [15]:
def analyze_optimization_results(traj, save_plots=True):
    """
    5.2 Implement result visualization and analysis tools
    5.3 Individual management - track hall of fame
    """
    if not L2L_AVAILABLE:
        print("L2L not available for analysis")
        return
    
    # Collect results
    generations = []
    fitness_values = []
    spike_counts = []
    parameters_history = []
    
    # 5.3 Create hall of fame tracking best neuron models
    hall_of_fame = []
    
    # Iterate through all runs
    for run_idx, run in enumerate(traj.f_iter_runs()):
        if hasattr(run, 'fitness') and hasattr(run.fitness, 'value'):
            fitness = run.fitness.value
            fitness_values.append(fitness)
            generations.append(run_idx)
            
            # Get features if available
            if hasattr(run, 'features'):
                spike_count = run.features.spike_count if hasattr(run.features, 'spike_count') else 0
                spike_counts.append(spike_count)
            
            # Collect parameters
            if hasattr(run, 'individual'):
                params = {}
                for group_name in ['membrane', 'channels', 'passive', 'dynamics']:
                    if hasattr(run.individual, group_name):
                        group = getattr(run.individual, group_name)
                        for param_name in dir(group):
                            if not param_name.startswith('_'):
                                params[f"{group_name}.{param_name}"] = getattr(group, param_name)
                parameters_history.append(params)
                
                # Add to hall of fame
                hall_of_fame.append({
                    'generation': run_idx,
                    'fitness': fitness,
                    'parameters': params,
                    'spike_count': spike_count if 'spike_count' in locals() else None
                })
    
    # Sort hall of fame by fitness
    hall_of_fame.sort(key=lambda x: x['fitness'])
    
    # Create visualizations
    if fitness_values and save_plots:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Fitness evolution
        ax = axes[0, 0]
        ax.plot(generations, fitness_values, 'b-', alpha=0.7)
        ax.set_xlabel('Generation')
        ax.set_ylabel('Fitness')
        ax.set_title('Fitness Evolution')
        ax.grid(True, alpha=0.3)
        
        # Spike count evolution
        if spike_counts:
            ax = axes[0, 1]
            ax.plot(generations[:len(spike_counts)], spike_counts, 'r-', alpha=0.7)
            ax.set_xlabel('Generation')
            ax.set_ylabel('Spike Count')
            ax.set_title('Spike Count Evolution')
            ax.grid(True, alpha=0.3)
        
        # Best fitness histogram
        ax = axes[1, 0]
        ax.hist(fitness_values, bins=30, alpha=0.7, color='green')
        ax.set_xlabel('Fitness')
        ax.set_ylabel('Frequency')
        ax.set_title('Fitness Distribution')
        
        # Parameter evolution for best individual
        if hall_of_fame and parameters_history:
            ax = axes[1, 1]
            best_params = hall_of_fame[0]['parameters']
            param_names = list(best_params.keys())[:5]  # Show first 5 parameters
            
            for i, param in enumerate(param_names):
                values = [gen_params.get(param, 0) for gen_params in parameters_history]
                ax.plot(generations[:len(values)], values, label=param.split('.')[-1])
            
            ax.set_xlabel('Generation')
            ax.set_ylabel('Parameter Value')
            ax.set_title('Parameter Evolution (Top 5)')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save figure
        analysis_dir = DIRS['analysis']
        fig_path = analysis_dir / f'optimization_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
        plt.savefig(fig_path)
        print(f"Results visualization saved to {fig_path}")
        plt.show()
    
    # Print hall of fame
    print("\n" + "="*60)
    print("HALL OF FAME - Top 5 Best Individuals")
    print("="*60)
    
    for i, individual in enumerate(hall_of_fame[:5]):
        print(f"\nRank {i+1}:")
        print(f"  Generation: {individual['generation']}")
        print(f"  Fitness: {individual['fitness']:.4f}")
        if individual['spike_count'] is not None:
            print(f"  Spike Count: {individual['spike_count']}")
        print("  Top parameters:")
        for param, value in list(individual['parameters'].items())[:3]:
            print(f"    {param}: {value:.6f}")
    
    return hall_of_fame

## Complete Example: Running L2L-Arbor Optimization

In [16]:
def run_complete_optimization(experiment_name="allen_neuron_l2l",
                            optimizer_type="evolution_strategies",
                            config_name="balanced",
                            n_iterations=50):
    """
    Complete example running the full L2L-Arbor optimization pipeline
    """
    if not L2L_AVAILABLE:
        print("L2L is not available. Install L2L to run optimization.")
        return
    
    print("Starting L2L-Arbor Optimization Pipeline")
    print("=" * 60)
    
    # 1. Setup experiment
    print("\n1. Setting up experiment...")
    experiment, traj = setup_experiment(experiment_name)
    
    if experiment is None:
        return
    
    # 2. Create optimizee
    print("\n2. Creating NeuronOptimizee...")
    optimizee = NeuronOptimizee(traj)
    
    # 3. Select and configure optimizer
    print(f"\n3. Configuring {optimizer_type} optimizer...")
    optimizer = select_optimizer(traj, optimizee, optimizer_type, config_name)
    
    if optimizer is None:
        print("Failed to create optimizer")
        return
    
    # 4. Run optimization
    print(f"\n4. Running optimization for {n_iterations} iterations...")
    print("This may take some time...")
    
    try:
        experiment.run_experiment(
            optimizee=optimizee,
            optimizer=optimizer,
            n_iterations=n_iterations,
            n_jobs=4  # Number of parallel jobs
        )
    except Exception as e:
        print(f"Error during optimization: {e}")
        return
    
    # 5. Finalize experiment
    print("\n5. Finalizing experiment...")
    experiment.end_experiment(optimizer)
    
    # 6. Analyze results
    print("\n6. Analyzing results...")
    hall_of_fame = analyze_optimization_results(traj)
    
    print("\n" + "="*60)
    print("Optimization complete!")
    print(f"Results saved in: {DIRS['results']}")
    print("="*60)
    
    return traj, hall_of_fame

In [17]:
# Run a test optimization (small scale)
if L2L_AVAILABLE:
    print("Running test optimization with 10 iterations...")
    test_traj, test_hof = run_complete_optimization(
        experiment_name="test_allen_opt",
        optimizer_type="evolution_strategies",
        config_name="balanced",
        n_iterations=10
    )
else:
    print("\nTo run the optimization, please install L2L:")
    print("pip install L2L")
    print("\nOr use the provided mock implementation for testing.")

Running test optimization with 10 iterations...
Starting L2L-Arbor Optimization Pipeline

1. Setting up experiment...
All output logs can be found in directory  /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_allen_opt_20250711_001620/logs
MainProcess utils.trajectory zq-Lenovo-Legion-R7000P2021H 239442 INFO    : Added new parameter group: runner_params
Runner parameters used: {'srun': '', 'exec': 'python3 "/home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_allen_opt_20250711_001620/simulation/run_optimizee.py"', 'max_workers': 32, 'work_path': '/home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_allen_opt_20250711_001620', 'paths_obj': <l2l.paths.Paths object at 0x7996af36b470>}
Experiment 'test_allen_opt_20250711_001620' configured successfully
Results will be stored in: /home/zq/code/gsoc-2025-l2l-arbor/l2l_arbor_project/results/test_allen_opt_20250711_001620

2. Creating NeuronOptimizee...
MainProcess utils.trajectory zq-Lenovo-Legion-R70

TypeError: 'ParameterDict' object is not callable